**Course outline:** Unit 03 | Notebook 07 | Python

# Get a district measles summary, one step at a time

Laila needs a short briefing: **How many measles case-event records are available for Kambia District, what onset period do they cover, and how complete are the onset dates?**

**Learning objective:** **Explore** district case-event records using software.

You have already distinguished individual case records from aggregate counts. Here you will retrieve individual records and turn them into a small summary. All Python is supplied. Run each code cell with **Shift+Enter**, read its output, then continue. You do not need to invent API commands or memorize IDs.

These are invented training records in a **browser-local simulation**. No connection to a health information system is needed. In this example, each event represents one case record; the count is **not** a count of laboratory-confirmed cases.

## The route through this notebook

1. Find the district: which records belong in Laila's briefing?
2. Read the dictionary: which field means rash onset?
3. Request the records for that district and surveillance program.
4. Read an ordinary table instead of a nested computer response.
5. Count records, count missing onset dates, and describe the known onset period.
6. Select **Submit**, then reflect on the missing date.

## A few terms before you start

| Familiar idea | Term used by the simulated system | What it means here |
| --- | --- | --- |
| Ask software for data | API request | A structured request, similar to specifying a filter in a reporting tool. |
| Read, rather than change, records | `api.get(...)` | The Python instruction that asks this simulation for a response. |
| A list you want to open | Endpoint, such as `/api/organisationUnits` | The name of a collection inside the simulation. It is not a website you need to visit. |
| District | Organisation unit | A location used to select records. |
| Dictionary field | Data element | A variable, such as Date of rash onset. |
| Internal code | ID | The computer's label for a district or variable. We look it up beside its readable name. |

Run the next collapsed setup cell once. It prepares the simulation; you do not need to read or edit its infrastructure code.


In [ ]:
import json
FIXTURE = json.loads('{"organisationUnits":[{"id":"OU_KAMBIA","displayName":"Kambia District"},{"id":"OU_PORT_LOKO","displayName":"Port Loko District"}],"dataElements":[{"id":"DE_ONSET","displayName":"Date of rash onset","valueType":"DATE"},{"id":"DE_AGE","displayName":"Age in years","valueType":"INTEGER"},{"id":"DE_CLASS","displayName":"Case classification","valueType":"TEXT"}],"events":[{"event":"EV_001","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-10-20","dataValues":[{"dataElement":"DE_ONSET","value":"2021-10-18"},{"dataElement":"DE_AGE","value":"4"},{"dataElement":"DE_CLASS","value":"suspected"}]},{"event":"EV_002","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-10-23","dataValues":[{"dataElement":"DE_ONSET","value":"2021-10-21"},{"dataElement":"DE_AGE","value":"8"},{"dataElement":"DE_CLASS","value":"suspected"}]},{"event":"EV_003","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-10-27","dataValues":[{"dataElement":"DE_ONSET","value":""},{"dataElement":"DE_AGE","value":"2"},{"dataElement":"DE_CLASS","value":"suspected"}]},{"event":"EV_004","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-11-02","dataValues":[{"dataElement":"DE_ONSET","value":"2021-10-30"},{"dataElement":"DE_AGE","value":"13"},{"dataElement":"DE_CLASS","value":"probable"}]},{"event":"EV_005","program":"P_MEASLES","orgUnit":"OU_KAMBIA","occurredAt":"2021-11-05","dataValues":[{"dataElement":"DE_ONSET","value":"2021-11-04"},{"dataElement":"DE_AGE","value":"6"},{"dataElement":"DE_CLASS","value":"confirmed"}]},{"event":"EV_006","program":"P_MEASLES","orgUnit":"OU_PORT_LOKO","occurredAt":"2021-10-25","dataValues":[{"dataElement":"DE_ONSET","value":"2021-10-24"},{"dataElement":"DE_AGE","value":"5"},{"dataElement":"DE_CLASS","value":"suspected"}]},{"event":"EV_007","program":"P_MEASLES","orgUnit":"OU_PORT_LOKO","occurredAt":"2021-11-03","dataValues":[{"dataElement":"DE_ONSET","value":"2021-11-01"},{"dataElement":"DE_AGE","value":"7"},{"dataElement":"DE_CLASS","value":"probable"}]}]}')
from copy import deepcopy

class CourseDhis2Adapter:
    """A read-only, in-memory subset of DHIS2-like GET responses."""

    def __init__(self, fixture):
        self._fixture = deepcopy(fixture)

    def get(self, path, params=None):
        params = dict(params or {})
        if path == "/api/organisationUnits":
            if params:
                raise ValueError("organisationUnits does not support query parameters in this pilot")
            return {"organisationUnits": deepcopy(self._fixture["organisationUnits"])}
        if path == "/api/dataElements":
            if params:
                raise ValueError("dataElements does not support query parameters in this pilot")
            return {"dataElements": deepcopy(self._fixture["dataElements"])}
        if path == "/api/tracker/events":
            if set(params) != {"program", "orgUnit"}:
                raise ValueError("events requires program and orgUnit")
            if params["program"] != "P_MEASLES":
                raise ValueError("unknown program")
            if params["orgUnit"] not in {u["id"] for u in self._fixture["organisationUnits"]}:
                raise ValueError("unknown organisation unit")
            return {"events": deepcopy([e for e in self._fixture["events"]
                                        if e["program"] == params["program"]
                                        and e["orgUnit"] == params["orgUnit"]])}
        raise ValueError("unsupported endpoint: " + str(path))

api = CourseDhis2Adapter(FIXTURE)

def show_missingness_matrix(frame, row_labels):
    import io
    import missingno as msno
    import matplotlib
    matplotlib.use("agg")
    import matplotlib.pyplot as plt
    from IPython.display import SVG, display
    fig, ax = plt.subplots(figsize=(7, 3.2))
    msno.matrix(frame, ax=ax, sparkline=False, labels=True, fontsize=10,
                color=(0.12, 0.46, 0.40))
    ax.set_yticks(range(len(frame)), labels=row_labels)
    ax.set_ylabel("Case-event record")
    ax.set_xlabel("Green = recorded; white = missing")
    ax.set_title("Completeness of the retrieved fields", pad=12)
    fig.tight_layout()
    output = io.BytesIO()
    fig.savefig(output, format="svg", bbox_inches="tight")
    svg = output.getvalue().decode("utf-8").replace("<svg ", '<svg role="img" aria-label="Completeness matrix. Green means recorded; white means missing. Exact counts are in the accompanying table." ', 1)
    display(SVG(data=svg))
    plt.close(fig)

def show_table(rows, columns):
    from html import escape
    from IPython.display import HTML, display
    headings = "".join("<th style='text-align:left;padding:0.4rem'>" + escape(label) + "</th>" for _, label in columns)
    body = []
    for row in rows:
        cells = []
        for key, _ in columns:
            value = row.get(key)
            text = "Missing" if value is None or value == "" else str(value)
            cells.append("<td style='padding:0.4rem;border-bottom:1px solid #ddd'>" + escape(text) + "</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML("<table><thead><tr>" + headings + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"))

def show_event_preview(event):
    from html import escape
    from IPython.display import HTML, display
    display(HTML('<details><summary>Optional: inspect one computer record</summary><pre style="white-space:pre-wrap">' + escape(json.dumps(event, indent=2)) + '</pre></details>'))

org_unit = events = onset_element = n_cases = n_missing_onset = first_onset = last_onset = None
print('Simulation ready. Continue with Step 1.')


## 1. Which district do we need?

Start with the **place** in Laila's question: Kambia District. The first request returns a list of districts and their internal codes.

`api.get(...)` asks for the list. The response contains a named list, `organisationUnits`; square brackets open that list. The supplied loop reads each district name and keeps the ID next to **Kambia District**. You can reuse this pattern without writing it yourself.

**Run the cell. Check:** Does the selected name say Kambia District? Its ID is just a computer label, not another geographic level.


In [ ]:
district_name = "Kambia District"
district_response = api.get("/api/organisationUnits")
districts = district_response["organisationUnits"]
show_table(districts, [("displayName", "District name"), ("id", "ID used in a request")])

org_unit = None
for district in districts:
    if district["displayName"] == district_name:
        org_unit = district["id"]
if org_unit is None:
    raise ValueError("Choose a district name from the table above.")
print("Selected district:", district_name, "| ID:", org_unit)


## 2. Which date answers the epidemiologic question?

Laila wants to describe **when illness began**, so use **Date of rash onset**. A data dictionary connects the readable field name, its internal ID, and its type. `DATE` means a date; the other fields here contain an age or a classification.

Run the cell to see the dictionary and select the onset field by its name. The pattern is the same as selecting the district in Step 1.

**Check:** The selected field should say Date of rash onset. A case-event date and a symptom-onset date are different concepts; we will compare them in Step 4.


In [ ]:
onset_label = "Date of rash onset"
dictionary_response = api.get("/api/dataElements")
data_elements = dictionary_response["dataElements"]
show_table(data_elements, [("displayName", "Field name"), ("id", "Field ID"), ("valueType", "Type")])

onset_element = None
for element in data_elements:
    if element["displayName"] == onset_label:
        onset_element = element["id"]
if onset_element is None:
    raise ValueError("Choose a field name from the dictionary above.")
print("Selected field:", onset_label, "| ID:", onset_element)


## 3. Ask for the right records

Now specify **what** and **where**:

| Request setting | Meaning |
| --- | --- |
| `program: P_MEASLES` | The measles surveillance program in this simulation. This code is supplied for you. |
| `orgUnit: org_unit` | The district ID you selected in Step 1. |

Both settings are needed. A district filter alone would not say which surveillance program to use. The response has a list named `events`; the next cell opens it and tells you how many records came back. An optional preview shows one raw computer record. You do not need to decode it yourself; Step 4 turns the selected field into a readable column.

**Check:** The printed district should still be Kambia. Retrieving records does not establish that they are complete, unique, or all confirmed cases.


In [ ]:
request_settings = {"program": "P_MEASLES", "orgUnit": org_unit}
print("Request settings:", request_settings)
event_response = api.get("/api/tracker/events", request_settings)
events = event_response["events"]
print("Retrieved", len(events), "case-event records for", district_name)
if events:
    show_event_preview(events[0])


## 4. Turn the response into a table you can read

The response stores each event's measurements inside a list called `dataValues`. Each measurement pairs a **field ID** with a **value**. This is a computer storage layout, not a new epidemiologic concept.

The supplied code opens each event, finds the field selected in Step 2, and puts its value into an onset-date column. The resulting table has **one row per case-event record**.

- **Event ID:** the record's identifier in this example.
- **Event date:** the date attached to the surveillance event; do not assume it means symptom onset or date of entry.
- **Rash onset date:** the field we selected from the dictionary.

**Before the summary:** Look down the onset column. Is any date missing? Compare an event date with its onset date. They need not be the same. A blank onset means unknown; it does not mean the event date is a valid replacement.


In [ ]:
onset_rows = []
for event in events:
    onset_value = ""
    for measurement in event["dataValues"]:
        if measurement["dataElement"] == onset_element:
            onset_value = measurement["value"]
    onset_rows.append({
        "event": event["event"],
        "event_date": event["occurredAt"],
        "onset": onset_value,
    })
show_table(onset_rows, [("event", "Event ID"), ("event_date", "Event date (not onset)"), ("onset", "Rash onset date")])


## See the missing date: a completeness matrix

The table says which record lacks an onset date. A **missingno matrix** makes that gap visible: each row is a record and each column is a field. **Green means a value is recorded; white means it is missing.** Use the row label to find the same record in Step 4.

Run this optional cell to draw the matrix. The first run loads the locally bundled plotting packages. The supplied code makes a **separate display copy**, turning blank strings into pandas' missing-value marker (`pd.NA`). Otherwise a blank string could look complete to the plotting library. The source records stay unchanged.

**Read the plot:** Which record has a missing rash-onset date? Is its event date also missing? Would a full green column prove that its dates are correct?

This chart shows **completeness**, one aspect of quality. A recorded date could still be wrong, a record could be duplicated, or a report could arrive late. Keep those questions separate. The table below gives exact counts as well as the visual pattern.


In [ ]:
# Run this supplied cell to visualize completeness; no edits are needed.
try:
    import missingno
except ImportError:
    import piplite
    await piplite.install(["missingno==0.5.2", "seaborn==0.13.2"])
import pandas as pd

# Normalize blanks only in a separate display copy, not in the source records.
quality_view = pd.DataFrame(onset_rows)[["event_date", "onset"]].replace(r"^\s*$", pd.NA, regex=True)
quality_view.columns = ["Event date", "Rash onset date"]
show_table([
    {"field": field, "missing": int(quality_view[field].isna().sum()), "total": len(quality_view)}
    for field in quality_view.columns
], [("field", "Field"), ("missing", "Missing values"), ("total", "Records")])
show_missingness_matrix(quality_view, [row["event"] for row in onset_rows])


## 5. Describe the records and their limitations

The summary answers three different questions:

| Measure | How the code gets it | What it tells Laila |
| --- | --- | --- |
| Case-event records | Count all retrieved records | How many records were selected, regardless of onset completeness. |
| Missing onset dates | Count blank onset values | How many records cannot be placed on an onset-based epidemic curve yet. |
| Earliest and latest known onset | Compare only nonblank onset dates | The onset period represented by known dates, not necessarily the full outbreak duration. |

`len` counts entries. `min` and `max` find the earliest and latest known dates here because these dates use the same `YYYY-MM-DD` format. We keep blank dates separate; we do not substitute the event date.

Run the cell and compare the summary with the table above. **A record with missing onset still contributes to the record count.**


In [ ]:
n_cases = len(events)
n_missing_onset = 0
known_onsets = []
for row in onset_rows:
    if row["onset"] == "":
        n_missing_onset += 1
    else:
        known_onsets.append(row["onset"])

first_onset = min(known_onsets) if known_onsets else None
last_onset = max(known_onsets) if known_onsets else None
show_table([
    {"measure": "Selected case-event records", "value": n_cases},
    {"measure": "Missing onset dates", "value": n_missing_onset},
    {"measure": "Earliest known onset", "value": first_onset},
    {"measure": "Latest known onset", "value": last_onset},
], [("measure", "Measure"), ("value", "Result")])


## 6. Submit and interpret

Run the next cell to display **Submit**. This checks the guided run for Kambia District. You can return to a step, rerun the later steps, and submit again.

In [ ]:
# Run to display Submit for your guided exploration.
try:
    import ipywidgets as widgets
except ImportError:
    import piplite
    await piplite.install("ipywidgets==8.1.7")
    import ipywidgets as widgets
from comm import create_comm
from html import escape
from IPython.display import display

def _show_dhis2_submit():
    button = widgets.Button(description="Submit", button_style="info", disabled=True)
    status = widgets.HTML(value="Connecting to your learning record...")
    channel = create_comm(target_name="al.submit", data={"cell": "1"})
    def receive(message):
        data = message["content"]["data"]
        if data.get("ready"):
            status.value = "Run Steps 1-5, then submit this guided exploration."
        elif data.get("outcome") == "Incomplete":
            status.value = "<b>Not complete yet.</b> Run each code cell in Steps 1-5, in order, then submit again. This attempt was recorded."
        elif data.get("outcome") == "Incorrect":
            status.value = "<b>Check the district, date field, and summary.</b> Laila needs Kambia District and Date of rash onset. Rerun Steps 1-5 in order, keeping missing onset dates separate from event dates. This attempt was recorded."
        elif data.get("outcome") == "Correct":
            status.value = "<b>Guided exploration complete.</b> Your selected records and summary match Laila's question. This attempt was recorded. Now explain the missing-date limitation in the reflection below."
        else:
            status.value = escape(data.get("error", "Could not record. Please submit again."))
        button.disabled = False
    channel.on_msg(receive)
    def submit(_):
        button.disabled = True
        status.value = "Checking and recording your guided exploration..."
        channel.send({"submit": True})
    button.on_click(submit)
    display(button, status)

_show_dhis2_submit()


## What would you tell the district team?

Run the next cell for reflection fields and select **Record** to keep your notes in this browser. These reflections are unscored.

In [ ]:
# Run this cell to write and record your reflections.
try:
    import ipywidgets as widgets
except ImportError:
    import piplite
    await piplite.install("ipywidgets==8.1.7")
    import ipywidgets as widgets
from html import escape
from IPython.display import display
from comm import create_comm

def _reflection_section(section, questions):
    prompts = [question["prompt"] for question in questions]
    fields = []
    for number, prompt in enumerate(prompts, 1):
        field = widgets.Textarea(placeholder="Write your reflection here", rows=5,
                                 layout=widgets.Layout(width="100%"))
        fields.append(field)
        display(widgets.VBox([
            widgets.HTML(value=f"<strong>{number}. {escape(prompt)}</strong>"), field
        ], layout=widgets.Layout(width="100%", padding="0.5rem", margin="0 0 0.5rem 0")))
    button = widgets.Button(description="Record", button_style="info", disabled=True)
    status = widgets.HTML(value="Connecting to your browser learner record...")
    status.add_class("course-reflection-status")
    channel = create_comm(target_name="al.reflections", data={"section": section})
    def receive(message):
        data = message["content"]["data"]
        if "entries" in data:
            entries = data["entries"]
            restored = 0
            earlier = 0
            for question, field in zip(questions, fields):
                saved = next((entry for entry in reversed(entries) if
                    (entry.get("id") == question["id"] and entry.get("version") == question["version"]) or
                    (question["version"] == 1 and not entry.get("id") and entry["prompt"] in question["legacyPrompts"])), None)
                field.value = saved["answer"] if saved else ""
                restored += saved is not None
                earlier += any(entry.get("id") == question["id"] and entry.get("version") != question["version"] for entry in entries)
            status.value = "Recorded reflections restored." if restored else "Select Record to save in this browser."
            if earlier or any(entry.get("history") for entry in entries) or len(entries) > restored:
                status.value += " Earlier responses remain in My learning record and Download reflections. Review them before answering revised questions."
        elif data.get("saved"):
            status.value = "Recorded in this browser. Your reflections are unscored."
        else:
            status.value = escape(data.get("error", "Could not record. Please try again."))
        button.disabled = False
    channel.on_msg(receive)
    def record(_):
        button.disabled = True
        status.value = "Recording..."
        channel.send({"entries": [{**question, "answer": field.value}
                                  for question, field in zip(questions, fields)]})
    def edited(_):
        status.value = "Changes not recorded. Select Record to save."
    for field in fields:
        field.observe(edited, names="value")
    button.on_click(record)
    display(button, status)

_reflection_section('exercise:u03-n07', [{'id': 'q1', 'version': 1, 'prompt': 'What does the summary tell Laila, and what can it not establish about the outbreak?', 'legacyPrompts': ['What does the summary tell Laila, and what can it not establish about the outbreak?']}, {'id': 'q2', 'version': 1, 'prompt': 'How could using an event date in place of missing rash onset affect an epidemic curve? What would you verify with the district team?', 'legacyPrompts': ['How could using an event date in place of missing rash onset affect an epidemic curve? What would you verify with the district team?']}])


## Continue Laila's investigation

For another practice run, change only `district_name` in Step 1 to **Port Loko District**, then rerun Steps 1-5. What changes in the returned records? Return it to **Kambia District** and rerun the steps before submitting Laila's task.

Next, open [My learning record](u03-n08-my-learning-record.ipynb) to review what your attempts recorded.


## References

- Public Health Informatics for FETP, course slides and practice exercises.
- [Unit 2 course slides (local PDF)](../../readings/ucsf/assets/unit-2-version-2-slides.pdf).
